# 🧠 Session 12 — Evaluation, Guardrails & Observability (LangSmith Focus)

---

### 🎯 **Focus:**
Testing, monitoring, and improving agent behavior using **LangSmith** for structured evaluation and observability.

### 🧩 **Key Concepts:**
- Evaluating LLM prompt strategies using LangSmith’s Eval Suite.
- Logging and monitoring runs with LangSmith dashboards.
- Adding guardrails via JSON Schema validation and scoring.
- Comparing prompt versions for accuracy and consistency.

---

## ⚙️ 1. Setup — Install and Configure LangSmith

We'll install the required packages and set up your environment for LangSmith tracking.


In [3]:
!pip install langchain langchain_openai langsmith



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 🔐 **Set Environment Variables**

In your terminal (or notebook), set your LangSmith credentials:


In [5]:
import os

os.environ["OPENAI_API_KEY"] = "open router api key"
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Session12_Eval"
os.environ["LANGCHAIN_API_KEY"] = "langsmith api key"


## 🧱 2. Define a Simple E-commerce Gift Advisor Agent

We’ll create two different **prompt strategies** for a “Gift Advisor Agent” —
one uses concise logic, and another is more verbose and emotional.

We’ll compare them using LangSmith’s evaluation tools.


In [6]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model="gpt-4o-mini",base_url=os.environ["OPENAI_BASE_URL"],api_key=os.environ["OPENAI_API_KEY"], temperature=0.7)

# --- Strategy A: Logical and concise ---
prompt_a = PromptTemplate.from_template("""
You are a gift advisor. Recommend a suitable gift based on the following:
Occasion: {occasion}
Recipient: {recipient}
Budget: {budget}

Respond briefly with one recommendation and a short reason.
""")

chain_a = LLMChain(llm=llm, prompt=prompt_a)

# --- Strategy B: Emotional and narrative ---
prompt_b = PromptTemplate.from_template("""
You are a thoughtful and empathetic gift consultant.
Given:
Occasion: {occasion}
Recipient: {recipient}
Budget: {budget}

Provide a heartfelt recommendation that connects emotionally with the recipient.
""")

chain_b = LLMChain(llm=llm, prompt=prompt_b)


C:\Users\Moonwalking\AppData\Local\Temp\ipykernel_4340\3867438041.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain_a = LLMChain(llm=llm, prompt=prompt_a)


## 🧠 3. Run a Few Test Cases

We’ll test both prompt strategies on the same dataset and record them in LangSmith for analysis.


In [7]:
test_cases = [
    {"occasion": "birthday", "recipient": "mom", "budget": "$50"},
    {"occasion": "anniversary", "recipient": "husband", "budget": "$100"},
    {"occasion": "graduation", "recipient": "friend", "budget": "$30"},
]

results_a, results_b = [], []

for case in test_cases:
    results_a.append(chain_a.invoke(case))
    results_b.append(chain_b.invoke(case))


## 📊 4. Log and Compare Results in LangSmith

LangSmith automatically tracks these runs (since tracing is enabled).
You can view all outputs, inputs, and model metadata in your LangSmith project dashboard.


In [8]:
# Results summary
for i, case in enumerate(test_cases):
    print(f"\n🎁 Test Case {i+1}: {case}")
    print("🧩 Strategy A:", results_a[i]["text"])
    print("💬 Strategy B:", results_b[i]["text"])



🎁 Test Case 1: {'occasion': 'birthday', 'recipient': 'mom', 'budget': '$50'}
🧩 Strategy A: I recommend a personalized photo album. It's a heartfelt gift that allows her to cherish and display her favorite memories, and you can find beautiful options within your budget. This thoughtful gesture shows your love and appreciation for her.
💬 Strategy B: For your mom's birthday, consider creating a personalized gift that reflects your love and appreciation for her. One heartfelt idea is to assemble a "Memory Jar." 

**What you’ll need:**
- A beautiful jar (you can find one at a craft store or repurpose a glass container).
- Colored paper or cardstock.
- A nice pen or markers.
- Ribbon or twine for decoration.

**Instructions:**
1. Cut the colored paper into small squares or strips.
2. Write down your favorite memories, reasons why you love her, or quotes that remind you of her on each piece of paper. Aim for at least 20-30 notes.
3. Fold each note and place it in the jar.
4. Decorate the jar

## 🧮 5. Evaluate Prompt Strategies in LangSmith

LangSmith lets you create **evaluation datasets** and run **Eval Suites** directly.

You can use their dashboard to:
- Score which strategy performed better (clarity, personalization, accuracy)
- Visualize the performance of different models or prompt styles
- Add human or model-based evaluators

👉 Go to: [https://smith.langchain.com/evals](https://smith.langchain.com/evals)


## 🧰 6. Adding Guardrails with JSON Schema Validation

We can enforce structured outputs for reliability using JSON Schema.


In [9]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

response_schemas = [
    ResponseSchema(name="gift", description="Suggested gift item"),
    ResponseSchema(name="reason", description="Short reason for the choice")
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = parser.get_format_instructions()

prompt_guard = PromptTemplate(
    template="""Recommend a gift in valid JSON format.
{format_instructions}

Occasion: {occasion}
Recipient: {recipient}
Budget: {budget}
""",
    input_variables=["occasion", "recipient", "budget"],
    partial_variables={"format_instructions": format_instructions},
)

guard_chain = LLMChain(llm=llm, prompt=prompt_guard)
result = guard_chain.invoke({"occasion": "wedding", "recipient": "colleague", "budget": "$75"})
parsed = parser.parse(result["text"])

print(parsed)


{'gift': 'Personalized cutting board', 'reason': 'A practical and thoughtful gift that adds a personal touch to their new kitchen.'}


## 📈 7. Observability — Viewing Results in LangSmith

Every chain run (inputs, outputs, tokens, timings, and intermediate steps)
is automatically visible in your LangSmith dashboard.

Visit your project page:
👉 [https://smith.langchain.com/projects](https://smith.langchain.com/projects)

From there, you can:
- Inspect each LLM call (prompt, output, latency)
- Compare strategies side-by-side
- Track dataset evaluations and metrics


# 🧾 8. Case Study — E-commerce Gift Advisor Accuracy Audit
### 🧩 Objective:
To audit the **accuracy and consistency** of the Gift Advisor Agent using LangSmith.
We’ll create a dataset of gift recommendation scenarios and use LangSmith to evaluate model performance across strategies.


In [10]:
# Step 1: Create a structured dataset of evaluation cases
dataset = [
    {"occasion": "birthday", "recipient": "dad", "budget": "$80", "ideal_gift": "A leather wallet or a set of golf accessories."},
    {"occasion": "wedding", "recipient": "friend", "budget": "$150", "ideal_gift": "A decorative home set or customized photo frame."},
    {"occasion": "baby shower", "recipient": "sister", "budget": "$60", "ideal_gift": "A baby care kit or personalized onesies."},
]

# Step 2: Run predictions using Strategy B (emotional tone)
audit_results = []

for case in dataset:
    prediction = chain_b.invoke(case)["text"]
    audit_results.append({
        "input": case,
        "prediction": prediction,
        "ground_truth": case["ideal_gift"]
    })

# Step 3: Display results
for r in audit_results:
    print("\n🎁 Case:", r["input"])
    print("🧠 Model Prediction:", r["prediction"])
    print("✅ Ideal Gift:", r["ground_truth"])



🎁 Case: {'occasion': 'birthday', 'recipient': 'dad', 'budget': '$80', 'ideal_gift': 'A leather wallet or a set of golf accessories.'}
🧠 Model Prediction: For your dad’s birthday, I recommend a personalized leather wallet, which beautifully combines practicality with a personal touch. You can find one within your budget of $80, especially if you look for options online or at local craft stores.

Choose a high-quality leather that feels good to the touch and ages beautifully over time. To make it truly special, consider having his initials or a meaningful date embossed on the wallet. This small detail will make it uniquely his and show that you put thought into the gift.

In addition to the wallet, you could include a heartfelt note expressing your gratitude for the lessons he's taught you and the memories you've shared. This could be a simple message about how he inspires you or a specific memory that makes you smile. 

This combination of a thoughtful gift and a personal message will 

### 🧮 Step 4: Score Accuracy

A simple keyword-matching accuracy check.
(LangSmith supports more advanced evaluations like model-based or human grading.)


In [11]:
scores = []
for r in audit_results:
    keywords = [w.lower() for w in r["ground_truth"].split()]
    match = sum(1 for w in keywords if w in r["prediction"].lower())
    score = round(match / len(keywords), 2)
    scores.append(score)
    print(f"🎯 Accuracy for '{r['input']['occasion']}': {score * 100}%")

overall = sum(scores) / len(scores)
print(f"\n📊 Overall Audit Accuracy: {overall * 100:.2f}%")

🎯 Accuracy for 'birthday': 67.0%
🎯 Accuracy for 'wedding': 38.0%
🎯 Accuracy for 'baby shower': 56.99999999999999%

📊 Overall Audit Accuracy: 54.00%


## 🎓 8. Summary

✅ **What You Learned:**
- How to evaluate and compare prompt strategies using **LangSmith**
- How to enable automatic observability in LangChain
- How to enforce structured JSON outputs with guardrails
- How to visualize evaluation results inside LangSmith dashboards

🎯 **Next Step:**
Try integrating LangSmith with your **LangGraph** or **CrewAI** workflows for end-to-end observability!
